# 05 · End-to-end — a population-driven MDC, provenance, and the SGWB

The finale chains everything: a **gwmock-pop** catalogue drives a
**signal + noise** simulation, we verify what came out, replay a run
**bit-for-bit from its metadata**, and generate a stochastic background
that we check against theory.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/isaac-cf-wong/20260706-leuven-gw-workshop-gwmock/blob/main/materials/notebooks/05-end-to-end.ipynb)

In [ ]:
# === Run this ONCE ===
# Installs gwmock on Google Colab / fresh kernels (needs Python 3.12-3.14).
# If gwmock is already available (e.g. you pre-installed it into a local
# venv), this skips the install and just reports the version.
# The [sgwb] extra pulls in the correlated-noise machinery used for the
# stochastic-background examples.
try:
    import gwmock
    print("gwmock already available:", gwmock.__version__)
except ModuleNotFoundError:
    %pip install -q "gwmock[sgwb]"

> **On Google Colab, the cell above may print red `pip` dependency-conflict
> warnings** about `gradio`, `numba`, or `huggingface-hub` wanting older
> `numpy` / `pydantic` / `typer`. Those are Colab's **preinstalled** packages,
> which this workshop never uses — **the gwmock install still succeeded, and
> you can ignore them.** (If a later cell ever errors with a `numpy`/ABI
> message, use *Runtime → Restart session* and re-run from the top.)

## 1. Sample a catalogue that fits our segment

Five GWTC-4-flavoured BBHs, with coalescence times pinned into a 128 s
window by overriding the `coa_time` node (GPS-safe float64 sampling is
the package default — see notebook 02):

In [ ]:
import numpy as np
from gwmock_pop import CBCSimulator, write_population_catalogue

sim = CBCSimulator(seed=11, parameters={
    "coa_time": {"sampler": {"function": "uniform",
                             "arguments": {"minimum": 1577491250.0,
                                           "maximum": 1577491330.0}}},
})
catalogue = sim.simulate(5)
write_population_catalogue("mdc_population.csv", catalogue)

order = np.argsort(np.asarray(catalogue["coa_time"]))
for i in order:
    print(f"event: m1={float(catalogue['detector_frame_mass_1'][i]):5.1f}  "
          f"m2={float(catalogue['detector_frame_mass_2'][i]):5.1f}  "
          f"d={float(catalogue['distance'][i]):7.1f} Mpc  "
          f"t={float(catalogue['coa_time'][i]):.1f}")

## 2. Simulate signals + noise from the catalogue

In [ ]:
%%writefile mdc.yaml
globals:
    simulator-arguments:
        sampling-frequency: 2048
        duration: 128
        total-duration: 128
        start-time: 1577491218
        seed: 42
    working-directory: .
    output-directory: output
    metadata-directory: metadata
orchestration:
    population:
        backend: FilePopulationLoader
        source-type: bbh
        arguments:
            path: mdc_population.csv
    signal:
        waveform-model: IMRPhenomXPHM
        minimum-frequency: 20
        earth-rotation: true
        detectors:
            - ET-2L-Aligned
        output:
            output_directory: signal
            file_name: 'E-{{ detectors }}_BBH-{{ start_time }}-{{ duration }}.gwf'
            arguments:
                channel: '{{ detectors }}:STRAIN'
    noise:
        arguments:
            psd_file: ET_10_full_cryo_psd
            seed: 42
        output:
            output_directory: noise
            file_name: 'E-{{ detectors }}_NOISE-{{ start_time }}-{{ duration }}.gwf'
            arguments:
                channel: '{{ detectors }}:STRAIN'


In [ ]:
!gwmock simulate mdc.yaml --author 'Workshop' --email 'you@example.org' --overwrite 2>&1 | tail -2

In [ ]:
import matplotlib.pyplot as plt
from gwpy.timeseries import TimeSeries

ch = "ET1_2L_ALIGNED_SARD:STRAIN"
sig = TimeSeries.read("output/signal/E-ET1_2L_ALIGNED_SARD_BBH-1577491218-128.gwf", channel=ch)

plt.figure(figsize=(10, 3))
plt.plot(sig.times.value - sig.t0.value, sig.value, lw=0.4)
for t in np.asarray(catalogue["coa_time"]):
    plt.axvline(t - sig.t0.value, color="r", ls=":", alpha=0.5)
plt.xlabel(f"time since GPS {sig.t0.value:.0f} [s]")
plt.ylabel("strain")
plt.title("five catalogue events in the ET1 signal channel (red: coalescence times)")
plt.show()

## 3. Merge and anchor the spectrum

Merge signal + noise into analysis strain, then check its ASD against the
noise curve we simulated with — the loud low-frequency inspirals aside,
the merged data should ride along the ET sensitivity:

In [ ]:
!gwmock merge \
    output/signal/E-ET1_2L_ALIGNED_SARD_BBH-1577491218-128.gwf \
    output/noise/E-ET1_2L_ALIGNED_SARD_NOISE-1577491218-128.gwf \
    --channel ET1_2L_ALIGNED_SARD:STRAIN --output strain_E1.gwf --force 2>&1 | tail -1

In [ ]:
from gwmock_noise.spectral import load_and_interpolate_psd

data = TimeSeries.read("strain_E1.gwf", channel=ch)
asd = data.asd(fftlength=8)
f = asd.frequencies.value
ref = load_and_interpolate_psd("ET_10_full_cryo_psd", f[1:])

plt.figure(figsize=(8, 4))
plt.loglog(f[1:], asd.value[1:], lw=0.7, label="merged strain ASD")
plt.loglog(f[1:], np.sqrt(ref), "k--", label="ET_10_full_cryo (input)")
plt.xlim(5, 1024)
plt.xlabel("frequency [Hz]")
plt.ylabel(r"ASD [$1/\sqrt{\rm Hz}$]")
plt.legend()
plt.grid(True, which="both", alpha=0.3)
plt.show()

## 4. Provenance — what the metadata records

Every run writes a JSON sidecar with the full config, seeds, package
versions, and per-file hashes:

In [ ]:
import json

meta = json.load(open("metadata/orchestration-0.metadata.json"))
print("top-level keys:", sorted(meta.keys()))
print()
print("gwmock version :", meta["gwmock_version"])
print("host           :", meta["host"]["platform"])
print("config sha256  :", meta["config_sha256"][:23], "...")
print("file hashes    :", len(meta.get("file_hashes", {})), "files")

In [ ]:
!gwmock validate output/ --metadata-paths metadata/ 2>&1 | tail -3

## 5. Reproduce a run from metadata alone

The metadata file is itself a runnable config: `gwmock simulate
<metadata.json>` regenerates the same data — including per-detector
templated outputs like the BBH run above (fixed in gwmock 0.9.0). We
demo it with an SGWB run, which doubles as our stochastic-background
example.

In [ ]:
%%writefile sgwb.yaml
globals:
    simulator-arguments:
        sampling-frequency: 512
        duration: 16
        total-duration: 16
        start-time: 1577491218
        seed: 42
    working-directory: .
    output-directory: output_sgwb
    metadata-directory: metadata_sgwb
orchestration:
    signal:
        source-type: sgwb
        detectors:
            - ET-2L-Aligned
        minimum-frequency: 5
        parameters:
            omega_ref: 1.0e-9
            spectral_index: 0.0
            reference_frequency: 25.0
        output:
            output_directory: signal
            file_name: sgwb-{{ counter }}.hdf5
            arguments:
                channel: '{{ detectors }}:SGWB'


In [ ]:
!gwmock simulate sgwb.yaml --overwrite 2>&1 | tail -2

In [ ]:
!mkdir -p repro && cp metadata_sgwb/orchestration-0.metadata.json repro/
!cd repro && gwmock simulate orchestration-0.metadata.json 2>&1 | tail -1

In [ ]:
import h5py

with h5py.File("output_sgwb/signal/sgwb-0.hdf5") as fa, \
     h5py.File("repro/output_sgwb/signal/sgwb-0.hdf5") as fb:
    key = list(fa.keys())[0]
    same = np.array_equal(fa[key][:], fb[key][:])
print("replayed dataset is bit-identical:", same)

## 6. Does the SGWB match theory? (anchor!)

Power-law energy density
$\Omega_{\rm GW}(f) = \Omega_{\rm ref} (f/f_{\rm ref})^{\alpha}$
maps to the one-sided strain PSD
$S_h(f) = \dfrac{3 H_0^2}{10\pi^2} \dfrac{\Omega_{\rm GW}(f)}{f^3}$.
Estimate the PSD of the generated data and overlay the theory curve —
an independent check of the generator:

In [ ]:
from scipy.signal import welch

with h5py.File("output_sgwb/signal/sgwb-0.hdf5") as fh:
    keys = list(fh.keys())
    a = fh[keys[0]][:]
    b = fh[keys[1]][:]

fs_sgwb = 512.0
omega_ref, alpha, fref = 1.0e-9, 0.0, 25.0
H0 = 67.66 * 1000 / 3.0856775814913673e22       # Planck-18, km/s/Mpc -> 1/s

fr, pxx = welch(a, fs=fs_sgwb, nperseg=2048, detrend=False)

def Sh(fq):
    out = np.zeros_like(fq)
    m = fq > 0
    out[m] = 3 * H0**2 / (10 * np.pi**2) * omega_ref * (fq[m] / fref) ** alpha / fq[m] ** 3
    return out

band = (fr >= 8) & (fr <= 200)
print(f"median measured/theory PSD ratio (8-200 Hz): {np.nanmedian(pxx[band] / Sh(fr)[band]):.2f}")
print(f"cross-channel Pearson correlation: {np.corrcoef(a, b)[0, 1]:.3f} (SGWB is correlated; noise would be ~0)")

plt.figure(figsize=(8, 4))
plt.loglog(fr[fr > 0], pxx[fr > 0], lw=0.6, label="measured (Welch)")
plt.loglog(fr[fr > 0], Sh(fr)[fr > 0], "k--", label=r"theory $S_h(f)$")
plt.xlim(5, 256)
plt.xlabel("frequency [Hz]")
plt.ylabel(r"$S_h(f)$ [1/Hz]")
plt.legend()
plt.grid(True, which="both", alpha=0.3)
plt.show()

## 7. Beyond the laptop

- **`gwmock batch`** — splits long configs into Slurm array jobs
  (`--job-name`, `--account`, `--time`, `--submit`).
- **`gwmock repository`** — create/upload/verify Zenodo datasets, so an
  MDC ships with its provenance.
- **Custom backends** — population, signal, and noise stages all accept
  third-party backends (`module:Class` or entry points): implement the
  protocol from `gwmock_signal.GWSimulator` / `gwmock_noise` /
  `gwmock_pop` and select it by name in the config.

## ✏️ Try it (remaining time)

1. Make it a **BNS** MDC: sample `BNSSimulator`, switch the waveform to
   `IMRPhenomPv2_NRTidalv2`, `minimum-frequency: 20` — note BNS signals
   are *long*; keep masses and duration modest.
2. Tilt the background: `spectral_index: 0.667` (the CBC-background
   slope) in `sgwb.yaml`, re-run the anchor plot.
3. Scale up *on paper*: what changes in `mdc.yaml` for one day of data?
   (`total-duration: 86400` — gwmock then writes a sequence of segments;
   do **not** run it here, it's ~GBs.)

---
## You've completed the session 🎉

You can now: install gwmock · run config-driven MDCs · sample population
models · build signals and noise from the Python engines · merge, validate
and **reproduce** datasets from metadata · and anchor simulated data
against theory.

Docs: <https://leuven-gravity-institute.github.io/gwmock>